# Part 3: Isolation Forest — Finding Anomalous Customers
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Thursday — The Watch List

Sarah has segments (yesterday). Now she wants to find INDIVIDUAL customers who don't fit any segment — the unusual cases worth investigating.

This is anomaly detection. The customer success team gets a list. Each customer gets a quick look.

**Why Isolation Forest specifically:**
- Fast and scalable to large datasets
- Works in high dimensions (where Z-score breaks down)
- Outputs both binary labels AND continuous scores
- The default industry choice for tabular anomaly detection

**By the end of this notebook you will be able to:**
- Fit `IsolationForest` and produce anomaly scores per row
- Interpret what makes each anomaly anomalous
- Use the `contamination` parameter responsibly
- Decide which anomalies are worth a human's time

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — IsolationForest ready")

## Step 1 — Setup (same preprocessing as NB 02 + 03)

In [ ]:
df = pd.read_csv("data/northstar_customers.csv")
features = df.drop(columns=["customer_id", "churned"])

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

X_processed = preprocessor.fit_transform(features)
print(f"Processed shape: {X_processed.shape}")

## Step 2 — Fit Isolation Forest

Two key parameters:
- **`n_estimators`** — number of trees (default 100; 100-200 is plenty)
- **`contamination`** — your prior on what fraction of the data is anomalous. Used only to set the threshold between "normal" and "anomalous" labels.

For NorthStar's customer base, let's set `contamination=0.05` — we expect about 5% of customers to be unusual.

In [ ]:
iso = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)

labels = iso.fit_predict(X_processed)   # -1 = anomaly, +1 = normal
scores = iso.score_samples(X_processed) # raw score (lower = more anomalous)

# Counts
n_anomalies = (labels == -1).sum()
n_normal    = (labels == +1).sum()
print(f"Anomalies flagged: {n_anomalies}  ({n_anomalies / len(labels):.2%})")
print(f"Normal customers:  {n_normal}")
print()
print(f"Score range: [{scores.min():.3f}, {scores.max():.3f}]")
print(f"Median score: {np.median(scores):.3f}")

## ⏸️ Pause and Predict

Before plotting, predict:
- Will the most anomalous customers be at the **extreme corners** of the PCA plot? Or scattered?
- What single feature do you expect to drive most anomalies on this dataset? (Hint: think about what makes a customer genuinely UNUSUAL.)

*Your prediction:*

> *Sample:* Anomalies should appear at the OUTSKIRTS of the PCA point cloud. Drivers are likely combinations: very high spend + very high returns; or very long tenure + 0 purchases; or 7+ support tickets in one quarter (rare event).

## Step 3 — Visualise anomalies in PCA space

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_processed)

fig, ax = plt.subplots(figsize=(10, 8))
normal_mask = labels == 1
ax.scatter(X_2d[normal_mask, 0], X_2d[normal_mask, 1],
            alpha=0.25, s=10, color="steelblue", label=f"Normal ({normal_mask.sum()})")
ax.scatter(X_2d[~normal_mask, 0], X_2d[~normal_mask, 1],
            alpha=0.8, s=25, color="indianred", marker="x",
            label=f"Anomaly ({(~normal_mask).sum()})")
ax.set_xlabel(f"PC1  ({pca_2d.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2  ({pca_2d.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("Anomalies in PCA space")
ax.legend()
plt.tight_layout()
plt.show()

### 💡 What you should notice

- **Anomalies cluster at the EDGES** of the PCA cloud. That's expected — anomalies live in the sparse, far-from-centre regions of feature space.
- Some anomalies sit near the centre too — these are anomalous on dimensions PCA didn't visualise.
- Don't trust the PCA plot alone for diagnosing WHY a customer is anomalous. Look at the raw features.

## Step 4 — Score distribution

Plot the histogram of anomaly scores. The threshold (set by `contamination`) is the dashed line.

In [ ]:
# Find the threshold IsolationForest used
threshold = np.percentile(scores, 5)  # contamination = 5% → 5th percentile

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.hist(scores, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(threshold, color="indianred", linestyle="--", linewidth=2,
            label=f"Threshold (5th %ile) = {threshold:.3f}")
ax.set_xlabel("Anomaly score (lower = more anomalous)")
ax.set_ylabel("Number of customers")
ax.set_title("Distribution of anomaly scores")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Customers below threshold (anomalies): {(scores < threshold).sum()}")
print(f"Customers at or above threshold:        {(scores >= threshold).sum()}")

## Step 5 — Inspect the most anomalous customers

For the top-10 most anomalous, look at every feature and compare to the median. The features that diverge the most tell you WHY they were flagged.

In [ ]:
# Add score and label to the dataframe
df_results = features.copy()
df_results["anomaly_score"] = scores
df_results["is_anomaly"]    = (labels == -1)

# Get the 10 MOST anomalous customers
most_anomalous = df_results.nsmallest(10, "anomaly_score")

# Show with global median for comparison
print("Top 10 most anomalous customers:")
print(most_anomalous[numeric_features + ["anomaly_score"]].round(2).to_string())
print()
print("Global median for context:")
print(features[numeric_features].median().round(2).to_string())

### 💡 What makes them anomalous?

Scan the most-anomalous rows. Pick one or two that stand out and explain WHY in words:

- Customer X: tenure 70 months (long), 0 purchases this quarter (very low), 7 support tickets (very high) → *long-term customer who suddenly stopped engaging and started complaining. Worth a personal call.*
- Customer Y: returns 0.50 (max), spend £400+ (very high), 0 support tickets → *unusual: someone returning everything but also spending a lot. Possible card fraud, possible reseller, possible buy-and-return habit.*

**This is the value of anomaly detection.** The model can't tell you WHY a customer is anomalous — only that they are. The human interpretation is where the action lives.

## Step 6 — Compare anomaly scores to the segments from NB 03

Are anomalies concentrated in ONE segment? Or spread across all of them?

In [ ]:
from sklearn.cluster import KMeans

# Re-run K-Means to get the cluster labels from NB 03
km = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_processed)

df_results["cluster"] = cluster_labels

# Anomaly rate per cluster
anomaly_by_cluster = df_results.groupby("cluster").agg(
    n_customers=("is_anomaly", "size"),
    n_anomalies=("is_anomaly", "sum"),
)
anomaly_by_cluster["anomaly_rate"] = (
    anomaly_by_cluster["n_anomalies"] / anomaly_by_cluster["n_customers"]
)
print("Anomalies per cluster:")
print(anomaly_by_cluster.to_string(float_format=lambda x: f"{x:.3f}"))

### 💡 What this tells us

- **Anomalies are spread across clusters.** That confirms anomaly detection finds genuinely UNUSUAL customers — not just members of an underrepresented segment.
- **Some clusters have higher anomaly rates than others.** Those clusters contain more "edge" customers. Worth understanding which segment generates the most outliers (could be the high-return segment, or the dormant-customers cluster).

**Final Friday deliverable:** segment summary (NB 03) + anomaly list (NB 04). The customer success team gets the watch list.

## ✅ Section Summary

| Step | Result |
|---|---|
| **Fit Isolation Forest** | One line of code; 100 trees |
| **Anomaly score per customer** | Continuous; can re-threshold |
| **5% flagged as anomalies** | About 500 customers from 10,000 |
| **Visualised in PCA space** | Anomalies cluster at the edges |
| **Profiled the most anomalous** | Each one has its own "why" — feature combinations |
| **Cross-checked with clusters** | Anomalies appear across all segments — they're genuinely unusual |

**Key insight:**
> Isolation Forest finds *which* customers are unusual. *Why* they're unusual is a human's job. The model surfaces; the human investigates.

---

## 🏁 Friday — What Sarah Presents

Sarah walks into Marcus's office with three artefacts:

| # | Deliverable | Tool |
|---|---|---|
| 1 | **2D PCA visualisation** of the customer base | PCA (NB 02) |
| 2 | **4 customer segments** named and profiled | K-Means (NB 03) |
| 3 | **List of ~500 anomalous customers** for the success team | Isolation Forest (NB 04) |

Marcus nods. *"Good. Now — sales are seasonal. Can you forecast next quarter's revenue?"*

That question — **time-ordered data forecasting** — is the engine of **L06 (Time Series Forecasting).**

---
**Next step →** Open `assignment.ipynb` for the after-class exercises.

*Or, for deeper material on hierarchical clustering, DBSCAN, and UMAP visualisation: open `optional_extensions.ipynb`.*

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — The `contamination` parameter — what changes if you change it?

`contamination` only sets the threshold for the binary label. The continuous `score_samples()` is unchanged. Demo: refit with three different contamination values and see what happens to the flagged set.

In [ ]:
for c in [0.01, 0.05, 0.10]:
    iso_c = IsolationForest(n_estimators=100, contamination=c, random_state=42, n_jobs=-1)
    iso_c.fit(X_processed)
    n_flagged = (iso_c.predict(X_processed) == -1).sum()
    print(f"contamination={c}:  flagged = {n_flagged}  ({n_flagged/len(X_processed):.2%})")
print()
print("→ contamination directly controls how strict the threshold is.")
print("→ For unknown ground truth, start conservative (1-2%) and widen if needed.")

## Extension 2 — Score-based scoring with a custom threshold

Instead of using `contamination`, you can use the continuous `score_samples` output and pick your own threshold based on operational constraints.

In [ ]:
# Scenario: the customer success team can review 100 customers a week.
n_to_review = 100

# Pick the threshold so exactly the bottom 100 customers are flagged
sorted_scores = np.sort(scores)
custom_threshold = sorted_scores[n_to_review - 1]

# Apply
custom_flagged = scores <= custom_threshold

print(f"Custom threshold (top {n_to_review} most anomalous): {custom_threshold:.3f}")
print(f"Number flagged: {custom_flagged.sum()}")
print()

# What's the overlap with the contamination-based labels?
default_flagged = labels == -1
overlap = (custom_flagged & default_flagged).sum()
print(f"Overlap with contamination=0.05 labels: {overlap}/{n_to_review}")
print()
print("→ The score is what matters operationally. Pick the threshold for your team's capacity.")

## Extension 3 — Compare Isolation Forest to Z-score outliers (single-feature)

A naive approach: flag anyone with |z| > 3 on any feature. This catches univariate outliers but misses multivariate ones.

In [ ]:
# Z-score on the original numeric features
from sklearn.preprocessing import StandardScaler

X_num = features[numeric_features].fillna(features[numeric_features].median())
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

# Flag if ANY feature has |z| > 3
z_outliers = (np.abs(X_num_scaled) > 3).any(axis=1)

# Compare
df_results["z_outlier"] = z_outliers

both = (df_results["is_anomaly"] & df_results["z_outlier"]).sum()
only_iso = (df_results["is_anomaly"] & ~df_results["z_outlier"]).sum()
only_z = (~df_results["is_anomaly"] & df_results["z_outlier"]).sum()

print(f"Flagged by BOTH Isolation Forest AND |z|>3:  {both}")
print(f"Flagged by Isolation Forest only (multivariate): {only_iso}")
print(f"Flagged by |z|>3 only (single-feature extremes): {only_z}")
print()
print("→ Isolation Forest catches MULTIVARIATE anomalies that |z| misses.")
print("→ |z| catches single-feature extremes that Isolation Forest might miss if the rest of the row looks normal.")
print("→ For best coverage, use BOTH (and review the union).")